# 🏛️ EXODUS-SPECULUM - Template Colab
> Pipeline de transformation vidéo vers clone 3D

---

## Instructions
1. Activer le GPU: `Runtime > Change runtime type > T4 GPU`
2. Monter Google Drive
3. Cloner le repo
4. Exécuter les sections dans l'ordre

## 🔧 Section 0: Setup & Configuration

In [ ]:
# Cell 1: Vérifier GPU
!nvidia-smi

import torch
print(f"\n🔥 PyTorch: {torch.__version__}")
print(f"🎮 CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎯 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Google Drive monté")
else:
    print("❌ Erreur montage Drive")

In [ ]:
# Cell 3: Cloner le repository EXODUS-SPECULUM
REPO_URL = "https://github.com/kioka8877-ux/-EXODUS-SPECULUM-.git"
REPO_DIR = "/content/-EXODUS-SPECULUM-"

import os
if os.path.exists(REPO_DIR):
    print("📁 Repo existe déjà, mise à jour...")
    !cd {REPO_DIR} && git pull
else:
    print("📥 Clonage du repo...")
    !git clone {REPO_URL} {REPO_DIR}

print(f"\n✅ Repo prêt: {REPO_DIR}")
!ls -la {REPO_DIR}

In [ ]:
# Cell 4: Créer structure SANCTUM sur Drive
# ============================================================
# STRUCTURE OFFICIELLE DES FRÉGATES
# ============================================================

ROOT_DRIVE = "/content/drive/MyDrive/EXODUS-SPECULUM"
SHARED_RESOURCES = "/content/drive/MyDrive/EXODUS_SHARED_RESOURCES"

FRIGATES = [
    "FRIGATE_00_CORTEX",
    "FRIGATE_01_SCANNER",
    "FRIGATE_02_SCENOGRAPHE",
    "FRIGATE_03_PROJECTIONNISTE",
    "FRIGATE_04_LOGISTIQUE",
    "FRIGATE_05_DIRECTEUR_PHOTO",
    "FRIGATE_06_ALCHIMISTE",
    "FRIGATE_07_PORTE_AVIONS",
]

SUBFOLDERS = ["CODEBASE", "INPUT", "OUTPUT"]

# Créer structure Frégates
for frigate in FRIGATES:
    for sub in SUBFOLDERS:
        path = f"{ROOT_DRIVE}/{frigate}/{sub}"
        os.makedirs(path, exist_ok=True)
    print(f"✅ {frigate}")

# Créer structure Shared Resources
ai_models = f"{SHARED_RESOURCES}/AI_MODELS/depth_anything_v2"
os.makedirs(ai_models, exist_ok=True)
print(f"\n✅ EXODUS_SHARED_RESOURCES créé")

print("\n🏛️ Structure SANCTUM créée sur Drive")

In [ ]:
# Cell 5: Installer dépendances
!pip install -q opencv-python-headless pillow tqdm

print("✅ Dépendances de base installées")

---

## 🔍 Section F01: FRÉGATE SCANNER
> Extraction de frames et estimation de profondeur

**Flux:**
```
Vidéo source: FRIGATE_01_SCANNER/INPUT/video.mp4
       ↓
Output: FRIGATE_01_SCANNER/OUTPUT/{project_id}/
       ├── frames/
       ├── depth_maps/
       └── spatial_data.json
```

In [ ]:
# Cell 6: Configuration Scanner
# ============================================================
# 🔧 MODIFIER CES PARAMÈTRES
# ============================================================

PROJECT_ID = "test_project_001"  # Identifiant unique du projet
VIDEO_FILENAME = "test_video.mp4"  # Nom du fichier dans INPUT/
EXTRACTION_FPS = 2.0  # Frames par seconde à extraire

# ============================================================
# CHEMINS SANCTUM (NE PAS MODIFIER)
# ============================================================

F01_INPUT = "/content/drive/MyDrive/EXODUS-SPECULUM/FRIGATE_01_SCANNER/INPUT/"
F01_OUTPUT = "/content/drive/MyDrive/EXODUS-SPECULUM/FRIGATE_01_SCANNER/OUTPUT/"
VIDEO_SOURCE = F01_INPUT + VIDEO_FILENAME

print("🔍 FRÉGATE SCANNER - Configuration")
print(f"   Project ID: {PROJECT_ID}")
print(f"   Vidéo INPUT: {VIDEO_SOURCE}")
print(f"   OUTPUT dir: {F01_OUTPUT}{PROJECT_ID}/")
print(f"   FPS extraction: {EXTRACTION_FPS}")

# Vérifier que la vidéo existe
import os
if os.path.exists(VIDEO_SOURCE):
    size_mb = os.path.getsize(VIDEO_SOURCE) / 1e6
    print(f"   ✅ Vidéo trouvée ({size_mb:.1f} MB)")
else:
    print(f"   ❌ Vidéo non trouvée!")
    print(f"   ➡️ Uploadez votre vidéo dans: {F01_INPUT}")

In [ ]:
# Cell 7: Lancer le Scanner (Extraction uniquement)
# ============================================================
import sys
sys.path.insert(0, '/content/-EXODUS-SPECULUM-/src')

from frigates.f01_scanner import FrameExtractor

output_dir = f"{F01_OUTPUT}{PROJECT_ID}"

# Extraction des frames
extractor = FrameExtractor(output_dir=output_dir, fps=EXTRACTION_FPS)
extraction_result = extractor.extract_frames(VIDEO_SOURCE)

print(f"\n📊 Résultat stocké dans `extraction_result`")

In [ ]:
# Cell 8: (Optionnel) Installer Depth Anything V2
# ============================================================
# Décommenter pour installer le modèle de profondeur

# !pip install -q git+https://github.com/DepthAnything/Depth-Anything-V2.git

# Télécharger le modèle ViT-Large (1.2GB) dans SHARED_RESOURCES
# DEPTH_MODEL_DIR = "/content/drive/MyDrive/EXODUS_SHARED_RESOURCES/AI_MODELS/depth_anything_v2"
# !mkdir -p {DEPTH_MODEL_DIR}
# !wget -O {DEPTH_MODEL_DIR}/depth_anything_v2_vitl.pth \
#     https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth

print("ℹ️ Décommenter les lignes ci-dessus pour installer Depth Anything V2")

In [ ]:
# Cell 9: Lancer le Scanner (Pipeline complet avec Depth)
# ============================================================
import sys
sys.path.insert(0, '/content/-EXODUS-SPECULUM-/src')

from frigates.f01_scanner import run_scanner

# Exécuter le pipeline complet
# output_base utilise F01_OUTPUT par défaut
result = run_scanner(
    video_path=VIDEO_SOURCE,
    project_id=PROJECT_ID,
    fps=EXTRACTION_FPS
)

print("\n📊 Résultat complet stocké dans `result`")

In [ ]:
# Cell 10: Visualiser les résultats
# ============================================================
import matplotlib.pyplot as plt
from PIL import Image
import os

# Charger les frames et depth maps
frames_dir = result['stages']['extraction']['output_dir']
depth_dir = result['stages']['depth'].get('output_dir', '')

if os.path.exists(frames_dir):
    frames = sorted([f for f in os.listdir(frames_dir) if f.endswith('.png')])[:3]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    for i, frame_name in enumerate(frames):
        # Frame originale
        frame_path = os.path.join(frames_dir, frame_name)
        frame_img = Image.open(frame_path)
        axes[0, i].imshow(frame_img)
        axes[0, i].set_title(f"Frame: {frame_name}")
        axes[0, i].axis('off')
        
        # Depth map correspondante
        if depth_dir and os.path.exists(depth_dir):
            depth_name = frame_name.replace('frame_', 'depth_')
            depth_path = os.path.join(depth_dir, depth_name)
            if os.path.exists(depth_path):
                depth_img = Image.open(depth_path)
                axes[1, i].imshow(depth_img, cmap='inferno')
                axes[1, i].set_title(f"Depth: {depth_name}")
                axes[1, i].axis('off')
            else:
                axes[1, i].text(0.5, 0.5, 'Depth N/A', ha='center', va='center')
                axes[1, i].axis('off')
        else:
            axes[1, i].text(0.5, 0.5, 'Depth skipped', ha='center', va='center')
            axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print(f"❌ Dossier frames non trouvé: {frames_dir}")

In [ ]:
# Cell 11: Afficher spatial_data.json
# ============================================================
import json

spatial_path = f"{F01_OUTPUT}{PROJECT_ID}/spatial_data.json"

if os.path.exists(spatial_path):
    with open(spatial_path, 'r') as f:
        spatial_data = json.load(f)
    print(json.dumps(spatial_data, indent=2))
else:
    print("⚠️ spatial_data.json non encore généré")

---

## 🧠 Section F02: FRÉGATE CORTEX (À venir)
> Intelligence IA pour l'analyse de scène

In [ ]:
# Placeholder pour F02-CORTEX
print("🚧 Section F02-CORTEX en développement...")

---

## 📐 Section F03-F07: Frégates suivantes (À venir)

- F02: SCÉNOGRAPHE (Génération géométrie)
- F03: PROJECTIONNISTE (Camera Projection)
- F04: LOGISTIQUE (Assets)
- F05: DIRECTEUR PHOTO (Camera)
- F06: ALCHIMISTE (Render)
- F07: PORTE-AVIONS (Output)

In [ ]:
# Fin du notebook
print("\n" + "=" * 60)
print("🏛️ EXODUS-SPECULUM - Session terminée")
print("=" * 60)